In [1]:
from pathlib import Path
import numpy as np
import torch
from scipy import signal
from scipy.integrate import simpson
from sklearn.ensemble import RandomForestRegressor

SEED = 1738
torch.manual_seed(SEED)
np.random.seed(SEED)

DATA = Path('eeg_starter/data')
FS, CH = 64, 14
BANDS = [(0.5, 4), (4, 8), (8, 12), (12, 30)]

def features(eeg):
    x = eeg[CH, :300 * FS].double().numpy()
    h1 = x.var()
    h2 = np.diff(x).var() / (x.var() + 1e-6)
    f, psd = signal.welch(x, FS, nperseg=FS)
    bp = [simpson(psd[(f >= lo) & (f <= hi)], dx=f[1] - f[0]) for lo, hi in BANDS]
    return np.array([h1, h2, *bp])

def load(name):
    seg = torch.load(DATA / f'segments_h1k_{name}.pt')
    X = np.stack([features(torch.load(DATA / 'pre_processed' / s[4].replace('.edf', '.pt'))) for s in seg])
    return X, np.array([s[3] for s in seg])

X_train, age_train = load('train')
X_val, age_val = load('val')

In [2]:
rf = RandomForestRegressor(n_estimators=200, max_depth=10, min_samples_leaf=5, criterion='squared_error', random_state=SEED, n_jobs=-1)
rf.fit(X_train, np.log(age_train))

pred = np.exp(rf.predict(X_val))
print(f"val RMSE {np.sqrt(np.mean((pred - age_val) ** 2)):.3f} years, MAE {np.mean(np.abs(pred - age_val)):.3f} years")

val RMSE 1.676 years, MAE 1.050 years
